# Named AutoDock type consumption — 2026-10-06

Bounded software qualification for DockingMT #43 and MolSysMT #222. Original provider rule/software attribution and parent atom axes are retained. Synthetic poses and real Vina admission do not establish docking readiness or predictive accuracy. Historical #42 receipts are preserved separately.

In [1]:
from pathlib import Path
import sys
assert Path(sys.prefix).name == 'molsyssuite@uibcdf_3.14'
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(root))
import json
import pandas as pd
import dockingmt as dmt
from devtools.qualify_named_types import qualify, save
record = qualify()
save(record)
print(record['interpreter'])
print(record['qualified_provider_commit'])

/home/liliana/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python
5bd893c85fe8d211663b2b1f865f5f1d2c382a90


## Independent chemical controls

Neutral/protonated amines, amide N, pyridine/pyrrole N and sulfur environments have explicit expected heavy-atom labels. All expected values are declared before provider evaluation. Charged source graphs and option mappings remain unchanged. Polygon/linear coordinates are declared test poses; they are not minimized conformers.

In [2]:
rows = []
for key, case in record['cases'].items():
    report = case['prepared']['metadata']['atom_type_assignment']
    rows.append({'case': key, 'role': case['role'], 'evaluated': report['n_atoms'], 'prepared': case['prepared']['n_atoms'], 'status': report['status'], 'assessment': case['preparation_assessment']['assessment'], 'parser': case['native_vina_parser_admitted']})
pd.DataFrame(rows)

,case,role,evaluated,prepared,status,assessment,parser
0,methanol_stages,partner,6,3,assigned,unassessed,True
1,CN:partner,partner,7,4,assigned,unassessed,True
2,CN:receptor,receptor,7,4,assigned,unassessed,True
3,C[NH3+]:partner,partner,8,5,assigned,unassessed,True
4,C[NH3+]:receptor,receptor,8,5,assigned,unassessed,True
5,CC(=O)N:partner,partner,9,6,assigned,unassessed,True
6,CC(=O)N:receptor,receptor,9,6,assigned,unassessed,True
7,c1ccncc1:partner,partner,11,6,assigned,unassessed,True
8,c1ccncc1:receptor,receptor,11,6,assigned,unassessed,True
9,c1cc[nH]c1:partner,partner,10,6,assigned,unassessed,True


## Parent chemical context survives fragment selection

The selected amide N retains its parent label N. Reclassifying the isolated fragment is not an equivalent operation. Provider complete coverage continues to refer to the original graph, while source maps identify the selected and retained inventory.

In [3]:
case = record['cases']['amide_parent_projection']
metadata = case['prepared']['metadata']
report = metadata['atom_type_assignment']
assert case['prepared']['atom_types'] == ['C', 'N']
assert report['n_atoms'] > case['prepared']['n_atoms']
print({key: report[key] for key in ('typing_scheme', 'method', 'rule_version', 'status', 'n_atoms', 'atom_source_indices')})
metadata['atom_type_projection']

{'typing_scheme': 'autodock4', 'method': 'chemical_environment', 'rule_version': 'chemical_environment@1', 'status': 'projected', 'n_atoms': 9, 'atom_source_indices': [0, 3]}


{'schema_version': '1.0',
 'policy': 'retain_provider_polar_h_merge_provider_nonpolar_h',
 'selected_source_atom_indices': [0, 3],
 'retained_selected_atom_indices': [0, 1],
 'retained_source_atom_indices': [0, 3],
 'prepared_atom_types': ['C', 'N'],
 'pdbqt_atom_indices': [0, 1],
 'pdbqt_to_source_atom_indices': [0, 3]}

## Explicit stages and indexed H

Typing follows requested fixed-state H addition and named charge assignment. No typing method, H inventory or charge model is chosen implicitly. The named provider profile distinguishes H/HD even for supported F/P parents; omitted nonpolar H still uses the existing charge transfer policy.

In [4]:
metadata = record['cases']['methanol_stages']['prepared']['metadata']
workflow = metadata['preparation_workflow']
assert workflow['stages'] == ['hydrogen_addition', 'partial_charge_assignment', 'atom_type_assignment']
assert workflow['prepared_to_input_atom_indices'] == [0, 1, None]
print(workflow['stages'], workflow['prepared_to_input_atom_indices'])
print({key: record['cases'][key]['prepared']['atom_types'] for key in ('F:polar_h', 'P:polar_h')})
metadata['atom_type_projection']

['hydrogen_addition', 'partial_charge_assignment', 'atom_type_assignment'] [0, 1, None]
{'F:polar_h': ['F', 'HD'], 'P:polar_h': ['P', 'HD', 'HD', 'HD']}


{'schema_version': '1.0',
 'policy': 'retain_provider_polar_h_merge_provider_nonpolar_h',
 'selected_source_atom_indices': [0, 1, 2, 3, 4, 5],
 'retained_selected_atom_indices': [0, 1, 5],
 'retained_source_atom_indices': [0, 1, 5],
 'prepared_atom_types': ['C', 'OA', 'HD'],
 'pdbqt_atom_indices': [0, 1, 2],
 'pdbqt_to_source_atom_indices': [0, 1, 5]}

## Original 181L BNZ

The same declared heavy-only benzene template precedes six local H additions, Gasteiger–Marsili charges and chemical_environment typing. Explicit intersection records B-factor/occupancy loss on the expanded copy. Experimental heavy-atom identity/pose and original native history remain source evidence; generated H geometry is modeled.

In [5]:
case = record['cases']['181l_bnz']
metadata = case['prepared']['metadata']
assert case['prepared']['atom_types'] == ['A'] * 6
assert metadata['atom_type_assignment']['n_atoms'] == 12
assert case['charge_audit']['assessment'] == 'consistent'
print(metadata['preparation_workflow']['hydrogen_addition']['n_added_hydrogens'])
print(metadata['preparation_workflow']['hydrogen_addition']['dropped_attributes'])
print(case['full_source_atom_indices'])
metadata['atom_type_projection']

6
['b_factor', 'occupancy']
[1299, 1300, 1301, 1302, 1303, 1304]


{'schema_version': '1.0',
 'policy': 'retain_provider_polar_h_merge_provider_nonpolar_h',
 'selected_source_atom_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 'retained_selected_atom_indices': [0, 1, 2, 3, 4, 5],
 'retained_source_atom_indices': [0, 1, 2, 3, 4, 5],
 'prepared_atom_types': ['A', 'A', 'A', 'A', 'A', 'A'],
 'pdbqt_atom_indices': [0, 1, 2, 3, 4, 5],
 'pdbqt_to_source_atom_indices': [0, 1, 2, 3, 4, 5]}

## Flexible written order

Two explicitly declared active bonds in unchanged 5X72 P59 exercise ROOT/BRANCH permutation. Every written label and original provider source index follows that permutation; branch counts are not used to infer atom identity or automatic torsion eligibility.

In [6]:
case = record['cases']['5x72_flexible']
projection = case['prepared']['metadata']['atom_type_projection']
order = projection['pdbqt_atom_indices']
assert order != list(range(case['prepared']['n_atoms']))
assert case['written_atom_types'] == [case['prepared']['atom_types'][i] for i in order]
pd.DataFrame({'written_index': range(len(order)), 'prepared_index': order, 'provider_source_index': projection['pdbqt_to_source_atom_indices'], 'type': case['written_atom_types']})

,written_index,prepared_index,provider_source_index,type
0,0,7,7,C
1,1,8,8,N
2,2,9,9,A
3,3,10,10,A
4,4,11,11,A
5,5,12,12,A
6,6,13,13,A
7,7,14,14,A
8,8,15,15,C
9,9,16,16,OA


## Nondefault units

The pm/coulomb context is admitted through public quantity conversion. Charges are audited in elementary charge and PDBQT coordinates in angstrom. Units do not choose or alter the named model.

In [7]:
case = record['cases']['nondefault_units']
assert case['charge_audit']['assessment'] == 'consistent'
print(case['prepared']['atom_types'], case['prepared']['charges'])
print(case['charge_audit']['charge_unit'])

['C', 'OA', 'HD'] [0.19000057917061752, -0.3996302435573072, 0.20962966438668965]
elementary_charge


## Real default Vina and saved-result provenance

A synthetic typed/charged receptor and methanol partner pass the default gate without exploratory provisional admission. One pose only proves backend interoperability. Scientific preparation assessment remains unassessed. Complete provider reports, consumer maps and captured exact input bytes survive finite JSON reload.

In [8]:
saved = record['default_vina_result']
assert len(saved['poses']) == 1
assert all(saved['provenance']['preparation'][role]['assessment'] == 'unassessed' for role in ('receptor', 'partner'))
assert dmt.verify_captured_inputs(saved['provenance']['backend_artifacts']) == record['verified_captured_bytes']
restored = dmt.DockingResult.from_dict(json.loads(json.dumps(saved, allow_nan=False)))
print('poses:', len(restored.poses))
print(record['verified_captured_bytes'])
print({role: saved['provenance']['preparation'][role]['metadata']['atom_type_assignment']['rule_version'] for role in ('receptor', 'partner')})

poses: 1
{'receptor': 887, 'partner': 910}
{'receptor': 'chemical_environment@1', 'partner': 'chemical_environment@1'}


## Negative controls and limits

Undeclared indexed H and changed prepared labels fail explicitly. Tests additionally cover stale native binding, manual label replacement, malformed evidence/maps and unchanged provider exception identity. Consumer guards detect accidental changes to saved labels/maps; they do not certify mutable molecular graphs or deliberate provenance rewriting.

In [9]:
print(record['negative_controls'])
assert json.loads(json.dumps(record, allow_nan=False)) == record
print('finite JSON:', len(record['cases']), 'cases')
for limit in record['limits']:
    print('-', limit)

{'virtual_h': {'exception': 'StructuralInconsistencyError', 'message': 'Structural inconsistency detected: AutoDock typing requires all hydrogens as indexed atoms; materialize the fixed inventory first.. Ensure that the atoms, residues, or frames match between the systems being compared or merged. Docs: https://www.uibcdf.org/MolSysMT'}, 'changed_prepared_label': {'exception': 'ArgumentError', 'message': "Invalid argument 'prepared': Prepared atom types no longer match their named assignment projection; prepare again with MolSysMT-assigned types."}}
finite JSON: 22 cases
- Local Python 3.14 source composition; no clean installed artifact, hosted or other Python/platform qualification.
- Synthetic polygon/linear poses are declared software controls, not conformers or experimental refinement.
- chemical_environment@1 is the provider experimental bounded profile, not a universal donor/acceptor model.
- Named type/charge declarations remove known provisional flags but assessment remains un

In [10]:
print(record['producer_versions'])
print(record['prior_source_profile'])
print('provider source files:', len(record['provider_source_sha256']))
print('consumer files:', len(record['consumer_implementation_sha256']))

{'molsysmt': '0.22.4+215.g5bd893c85', 'argdigest': '0.15.0+1.g5c6711e', 'molsysviewer': '0.23.4+90.gc046fca1', 'pyunitwizard': '0.28.1+3.g2ab37a5', 'smonitor': '0.19.0', 'depdigest': '0.13.0+3.g0568f9a', 'rdkit': '2026.3.1', 'vina': '1.2.7', 'numpy': '2.4.6'}
{'path': 'devguide/validation/data/chemical_templates/source_profile_2026-10-06.json', 'sha256': 'a201cf4b1429087cd714786b917a9f590b35bc8810ca5ee3e506971c8008236d'}
provider source files: 9
consumer files: 7
